## No memory

In [1]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm

from dotenv import load_dotenv
load_dotenv()

llm = create_azure_llm()

agent = create_agent(llm)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [2]:
from langchain.messages import HumanMessage
from pprint import pprint

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pprint(response["messages"][-1].content)

("Hello Ali! It's great to meet you. Green is such a refreshing and calming "
 'color. Do you have a particular shade of green that you like the most?')


In [3]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response["messages"][-1].content)

"I don't know your favourite colour yet. Would you like to tell me?"


## Memory

In [4]:
# langgraph checkpoint API.  Snapshot of the graph state at a given point in time. 
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),  
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [6]:
pprint(response["messages"][-1].content)

("Hello Ali! It's nice to meet you. Green is a wonderful color—so calming and "
 'refreshing. How can I assist you today?')


In [7]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response["messages"][-1].content)

'Your favourite colour is green!'


In [8]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pprint(response["messages"][-1].content)

# Optional: inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pprint(state.values["messages"][-4:])

('You live in Lahore, and your loyalty tier is gold. How can I assist you '
 'further, Ali?')
[HumanMessage(content="Store this custom data in memory and use it for future answers:\n{'name': 'Ali', 'favorite_color': 'green', 'city': 'Lahore', 'loyalty_tier': 'gold'}", additional_kwargs={}, response_metadata={}, id='a97adc11-5f62-46d5-b0cf-1d96d3fca879'),
 AIMessage(content="Got it, Ali! I've stored the following information:\n- Name: Ali\n- Favorite color: green\n- City: Lahore\n- Loyalty tier: gold\n\nI'll use this information to personalize my responses in the future. How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 116, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.

In [ ]:
def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    return result["messages"][-1].content

Scenario 1A: You prefer PostgreSQL as your database.
Scenario 1B: You prefer MongoDB.


In [11]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = {"configurable": {"thread_id": "memory-demo-A"}}
config_b = {"configurable": {"thread_id": "memory-demo-B"}}

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

print("Scenario 1A:", ask(config_a, "What database do I prefer?"))
print("Scenario 1B:", ask(config_b, "What database do I prefer?"))

Scenario 1A: You prefer PostgreSQL as your database.
Scenario 1B: You prefer MongoDB.


In [12]:
# Scenario 2: Preference update within the same thread
config_update = {"configurable": {"thread_id": "memory-demo-update"}}

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
print("Scenario 2:", ask(config_update, "Which deployment region should you use now?"))

# Scenario 3: Multi-step task memory
config_task = {"configurable": {"thread_id": "memory-demo-task"}}

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
print("Scenario 3A:", ask(config_task, "Summarize my constraints in one sentence."))
print("Scenario 3B:", ask(config_task, "Given those constraints, suggest 3 practical next steps."))

# Optional: inspect latest stored messages for one scenario
task_state = agent.get_state(config_task)
print("\nRecent messages in Scenario 3 thread:")
pprint(task_state.values["messages"][-4:])

Scenario 2: You should use the **West Europe** deployment region now, based on your latest update.
Scenario 3A: You need to develop a chatbot MVP with essential features by Friday, working within a $2,000 budget.
Scenario 3B: Here are three practical next steps given your Friday deadline and $2,000 budget:

1. **Clearly define the chatbot’s primary goal and must-have features** to keep the scope focused and achievable within the time and budget.  
2. **Select an affordable, easy-to-use chatbot platform or framework** (such as Dialogflow, ManyChat, or Tars) that enables rapid development without heavy coding.  
3. **Compile and organize any necessary content or FAQs upfront** to streamline chatbot training or scripting and avoid delays during implementation.

Recent messages in Scenario 3 thread:
[HumanMessage(content='Summarize my constraints in one sentence.', additional_kwargs={}, response_metadata={}, id='59e2a5a6-4501-4725-868b-fa756b72837b'),
 AIMessage(content='You need to develo